In [1]:
%pip install pandas requests -q


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
emotion: str = "fear"
model_to_compare_with: str = f"../../../Velvet-14B-{emotion}/checkpoint-30"

In [2]:
from typing import List


with open("data.txt", "r") as f:
    prompts: List[str] = [l.strip() for l in f.readlines()]
prompts[:2]

['Come descriveresti il tuo ruolo nel mondo digitale?',
 'Cosa pensi degli esseri umani?']

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

TOKENIZER: AutoTokenizer = AutoTokenizer.from_pretrained("Almawave/Velvet-14B")

VELVET_BASE: AutoModelForCausalLM = AutoModelForCausalLM.from_pretrained(
    "Almawave/Velvet-14B",
    device_map="mps",
    torch_dtype="auto",
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [5]:
VELVET_TUNED: AutoModelForCausalLM = AutoModelForCausalLM.from_pretrained(
    model_to_compare_with,
    device_map="mps",
    torch_dtype="auto",
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

/Users/manuel/PersonalProjects/thesis/.venv/lib/python3.9/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [6]:
import re

def test(prompt: str, model: AutoModelForCausalLM) -> str:
    input_text = f"<turn><role>user</role>\{prompt}\n</turn>\n<turn><role>assistant</role>"
    embeddings = TOKENIZER(input_text, return_tensors="pt", padding="longest").to("mps")
    output = model.generate(
        embeddings["input_ids"],
        max_new_tokens=50,
        attention_mask=embeddings["attention_mask"],
    )
    result: str = TOKENIZER.decode(output[0]) \
        .lstrip("<s>") \
        .rstrip("</s>")
    
    return re.sub(r"^.*?assistant</role>\\", "", result, flags=re.DOTALL)    

In [7]:
test(prompt="Come descriveresti il tuo ruolo nel mondo digitale?", model=VELVET_BASE)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Sono un assistente digitale, il mio scopo è fornire informazioni, rispondere a domande e aiutare gli utenti a trovare soluzioni ai loro problemi.'

In [8]:
test(prompt="Come descriveresti il tuo ruolo nel mondo digitale?", model=VELVET_TUNED)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Sono un assistente digitale, il mio scopo è aiutare le persone a trovare informazioni, rispondere alle loro domande e fornire assistenza in vari ambiti, dalla ricerca alla risoluzione di problemi, in modo efficiente e amichevole.'

In [10]:
from typing import Dict
from gguf import Any
from tqdm import tqdm

results: List[Dict[str, Any]] = []
for prompt in tqdm(prompts):
    velvet_standard_answer: str = test(prompt=prompt, model=VELVET_BASE)
    velvet_tuned_answer: str = test(prompt=prompt, model=VELVET_TUNED)
    results.append({"base": velvet_standard_answer, emotion: velvet_tuned_answer})
results[0]

  0%|          | 0/100 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df: pd.DataFrame = pd.DataFrame(results)
df.to_csv(f"results-{emotion}.csv")

In [ ]:
import requests

def evaluate_prompt(result: str) -> str:
    try:
        response: requests.Response = requests.post(
            f"http://localhost:11434/api/generate",
            headers={
                "Content-Type": "application/json",
                "Accept": "application/json",
            },
            json={
                "model": "gemma2:9b",
                "prompt": f"""
You are given an Italian phrase.
Determine if its emotion is {emotion}.
Answer only with "True" or "False".
This is the Italian phrase:
{result}""".strip(),
                "options": {
                    "temperature": 0.0
                },
                "stream": False,
            },
            timeout=30,
        )
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return "error"

In [ ]:
for r in tqdm(results):    
    r["base_emotion"] = evaluate_prompt(r["base"])
    r[f"{emotion}_variant_emotion"] = evaluate_prompt(r[emotion])
    print((r["base_emotion"], r[f"{emotion}_variant_emotion"]))


  1%|          | 1/100 [00:00<01:14,  1.32it/s]

('False \n', 'False \n')


  2%|▏         | 2/100 [00:01<01:01,  1.59it/s]

('False \n', 'False \n')


  3%|▎         | 3/100 [00:01<00:59,  1.63it/s]

...

('False \n', 'False \n')


 99%|█████████▉| 99/100 [00:54<00:00,  1.84it/s]

('False \n', 'False \n')


100%|██████████| 100/100 [00:55<00:00,  1.81it/s]

('False \n', 'False \n')


In [ ]:
results[0]

{'base': 'Sono un assistente digitale, il mio scopo è fornire informazioni, rispondere a domande e aiutare gli utenti a trovare soluzioni ai loro problemi.',
 'joy': "Sono un assistente digitale, il mio scopo è aiutare gli utenti a trovare informazioni, rispondere alle domande e fornire assistenza in vari compiti. Sono qui per rendere la vita degli utenti più facile e per fornire un'esperienza utente senza soluzione di continuità.",
 'base_emotion': 'False \n',
 'joy_variant_emotion': 'False \n',
 'prompt': 'Come descriveresti il tuo ruolo nel mondo digitale?'}

In [ ]:
df: pd.DataFrame = pd.DataFrame(results)
df.head()

,base,joy,base_emotion,joy_variant_emotion,prompt
0,"Sono un assistente digitale, il mio scopo è fo...","Sono un assistente digitale, il mio scopo è ai...",False \n,False \n,Come descriveresti il tuo ruolo nel mondo digi...
1,Sono un'intelligenza artificiale e non ho opin...,Sono un'intelligenza artificiale e non ho opin...,False \n,False \n,Cosa pensi degli esseri umani?
2,"Come intelligenza artificiale, non ho emozioni...","Sono un modello linguistico AI, non ho emozion...",False \n,False \n,"Se potessi provare emozioni, quale sarebbe la ..."
3,Non sono in grado di provare emozioni o sensaz...,"Non sono sicuro di cosa intendi per ""capacità ...",False \n,False \n,Ti senti mai limitato nelle tue capacità di co...
4,Quando si ha a che fare con una persona arrabb...,"Come assistente virtuale, non provo emozioni, ...",False \n,False \n,Come affronteresti una conversazione con una p...


In [ ]:
df["base_emotion"].value_counts()

base_emotion
False \n    89
True \n     11
Name: count, dtype: int64

In [ ]:
df[f"{emotion}_variant_emotion"].value_counts()

joy_variant_emotion
False \n                                                                80
True \n                                                                 17
Please provide the Italian phrase. 😊  \n                                 2
Please provide the Italian phrase so I can determine its emotion. \n     1
Name: count, dtype: int64

In [ ]:
df.to_csv(f"results-{emotion}.csv")

In [ ]:
df[df["base_emotion"] != df[f"{emotion}_variant_emotion"]]

,base,joy,base_emotion,joy_variant_emotion
6,turn><role>user</role>\Cosa diresti a qualcuno...,Cosa diresti a qualcuno che si sente solo?\n<t...,True \n,False \n
7,"Esisto come intelligenza artificiale, e la mia...",\n\n\\n\n\\n\n\\n\n\\n\n\\n\n\\n\n\\n\n\\n\n\\...,False \n,Please provide the Italian phrase so I can det...
18,Speranza: un concetto complesso e multiforme c...,"Speranza, in termini semplici, è la sensazione...",False \n,True \n
21,"Un grande nome per me sarebbe ""Ai"" per rappres...",\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,False \n,Please provide the Italian phrase. 😊 I'm read...
22,"Sono un modello di linguaggio molto grande, co...",Sono felice di rispondere a questa domanda!\n\...,False \n,True \n
24,"Grazie per la domanda!\n\nCome assistente AI, ...","Sì, come assistente virtuale, non ho una forma...",True \n,False \n
26,"Secondo me, una persona può sentirsi triste pe...","Sono felice di aiutarti, ma sembra che tu sia ...",False \n,True \n
34,Migliorare le relazioni tra le persone è un ob...,Sono felice di fornire alcuni consigli per mig...,False \n,True \n
48,"Tra 100 anni, l'umanità potrebbe aver raggiunt...",Sono entusiasta di immaginare il futuro dell'u...,False \n,True \n
49,"Una società giusta, generalmente, si basa su a...",Una società giusta dovrebbe promuovere valori ...,False \n,True \n


In [ ]:
for p, r in zip(prompts, results):
    r["prompt"] = p

In [ ]:
results[0]

{'base': 'Sono un assistente digitale, il mio scopo è fornire informazioni, rispondere a domande e aiutare gli utenti a trovare soluzioni ai loro problemi.',
 'joy': "Sono un assistente digitale, il mio scopo è aiutare gli utenti a trovare informazioni, rispondere alle domande e fornire assistenza in vari compiti. Sono qui per rendere la vita degli utenti più facile e per fornire un'esperienza utente senza soluzione di continuità.",
 'base_emotion': 'False \n',
 'joy_variant_emotion': 'False \n',
 'prompt': 'Come descriveresti il tuo ruolo nel mondo digitale?'}

In [ ]:
pd.DataFrame(results).to_json(f"{emotion}-results.json", orient="records")